# Day 16 · 训练日志与踩坑

**配套讲义**: [`days/day-16.md`](../days/day-16.md) ｜ **需要 GPU（云机器）**

把 trainer log 变成三联图（loss / lr / grad_norm），**并主动制造一个 bug 再修好** —— 后者才是今天真正的产出。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w3.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys, torch
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.version.cuda, "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"gpu    : {p.name}  {p.total_memory / 1024**3:.0f} GB")
    print("bf16   :", torch.cuda.is_bf16_supported())
else:
    print("⚠️  没有 GPU —— 这一天的训练/推理跑不了。先看 docs/13-hardware-and-cost.md 租机器")

## 1. 解析日志并画三联图

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.train.monitor",
                    "outputs/qwen25vl3b-cx-lora-v0"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout or r.stderr)

## 2. 反例：人为造一个坏日志，训练你的诊断直觉

下面直接构造三段「病态曲线」，让 `diagnose()` 判一遍。
你要做的是**先自己猜病因，再看脚本的判断**。

In [ ]:
import sys; sys.path.insert(0, "..")
from src.train.monitor import diagnose

cases = {
    "病态A": [{"step": i, "loss": 2.5 + 0.001 * i, "grad_norm": 0.9, "learning_rate": 3e-5}
              for i in range(100)],
    "病态B": [{"step": i, "loss": 2.5 if i < 10 else float("nan"),
               "grad_norm": 0.9, "learning_rate": 3e-5} for i in range(100)],
    "病态C": [{"step": i, "loss": 1.0 if i < 50 else 0.2,
               "grad_norm": 12.0 if i % 17 == 0 else 0.8,
               "learning_rate": 3e-5} for i in range(100)],
}
for name, log in cases.items():
    print("=" * 60)
    print(name)
    try:
        print(diagnose(log))
    except Exception as e:
        print("diagnose 需要不同的输入结构 →", e)

## 3. 今日的 bug 实验（必做）

三种改坏方式，选一种真的改，跑 50 步，观察症状：

| 改坏方式 | 症状 |
|---|---|
| chat template 换成错的 | loss 降得下去，但**推理时胡说**（最阴的一种） |
| label mask 包含 user 段 | 模型学会复述问题，loss 虚低 |
| lr 调大 10 倍 | loss 抖动/爆炸，grad_norm 尖刺 |

把「症状 → 根因 → 修法」三行写进打卡 —— 这三行比三联图值钱。

In [ ]:
bug_note = """
症状：
根因：
修法：
"""
print(bug_note)

## 验收清单

- [ ] 三联图已生成，并写了一段**解读**（不是贴图了事）
- [ ] **主动制造并修复了至少 1 个 bug**，能说清「症状 → 判据 → 根因 → 修法」
- [ ] 能指出你这次训练的过拟合拐点大概在第几步（或说明为什么还没到）
- [ ] 知道 `eval_loss` 抬头不一定是 bug —— 要先看 `train_loss` 是否还在降

**卡住了？** 回看 [`days/day-16.md`](../days/day-16.md) 第五节「容易踩的坑」。

> **明天**：`days/day-17.md` —— DeepSpeed / FSDP 配置（本地就能做完）